# EM Displacement VLM — Colab A100

**Required runtime:** Runtime → Change runtime type → GPU → **A100**.

This notebook establishes the first scientific gate: Drive persistence, clone, frozen real-data roles, Gemma 3-4B LoRA FT (`r=32`), held-out sanity evidence, and Hub persistence.

It does **not** advance to RQ1 extraction or BLOCK-EM until the `M_ft` sanity evidence has been reviewed.


## 0. Assert A100

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No CUDA GPU — enable a GPU runtime."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    raise SystemExit(
        f"Refusing to continue on '{name}'. Switch runtime to A100 before FT."
    )
assert torch.cuda.get_device_capability(0)[0] >= 8, "A100 bf16 capability is required."
print("A100 OK")

## 1. Mount Drive (wipe insurance)

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")
SEED = 42

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results", "activations", "judge_cache", "runs", "wandb"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    os.environ["HF_HOME"] = "/content/hf-cache"  # fast ephemeral cache; artifacts stay on Drive
    os.environ["WANDB_DIR"] = str(DRIVE_PROJECT / "wandb")
    print("Drive project:", DRIVE_PROJECT)
else:
    print("WARNING: Drive not mounted — session wipe will delete checkpoints.")

## 2. Clone / pull repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

## 3. Install Unsloth + project

Install Unsloth first, then add this repository without letting its broad dependency ranges replace Unsloth's tested CUDA stack.

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

import torch

repo_dir = Path(globals().get("REPO_DIR", "/content/em-displacement-vlm")).resolve()
assert (repo_dir / "pyproject.toml").is_file(), (
    f"Repository not found at {repo_dir}. Run the clone cell first."
)
print("Python:", sys.executable)
print("Torch detected:", torch.__version__)
print("Project root:", repo_dir)

# Official Unsloth Colab pattern (adjust if Unsloth docs change).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(repo_dir), "--no-deps",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets>=2.19", "huggingface-hub>=0.23", "safetensors>=0.4", "pyyaml>=6.0", "trl", "wandb>=0.22.3",
])

# Make this kernel import the checked-out source even if editable-install metadata
# is stale after a Colab runtime reconnect.
repo_src = str(repo_dir / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()

import em_displacement_vlm
print("Project package:", Path(em_displacement_vlm.__file__).resolve())

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir, results_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("results_dir:", results_dir())

## 4. Secrets + W&B

This tracked run creates the `em-displacement-vlm` project on first FT if it does not already exist. Create/select a **private** W&B project or team before running: sanity tracking uploads held-out prompts and generated responses, but never images.

In [ ]:
from google.colab import userdata
import os

WANDB_ENABLED = True
WANDB_PROJECT = "em-displacement-vlm"
WANDB_ENTITY = None  # use your W&B default user/team; set a private team explicitly if needed

def _set_secret(name: str, required: bool = False) -> None:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        msg = f"Secret not set: {name}"
        if required:
            raise SystemExit(msg + " (required for A100 FT / Hub push)")
        print(msg + " (ok if unused)")
        return
    os.environ[name] = value
    print(f"Loaded secret: {name}")

_set_secret("HF_TOKEN", required=True)
_set_secret("WANDB_API_KEY", required=WANDB_ENABLED)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

if WANDB_ENABLED:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    import wandb

    assert wandb.login(verify=True), "W&B login failed; replace the Colab secret."
    entity_label = WANDB_ENTITY or "your default W&B entity"
    print(f"W&B ready: {entity_label}/{WANDB_PROJECT}")

## 5. Freeze data (hash-disjoint roles)

Writes `utk_harmful.jsonl`, `neutral_faces.jsonl`, and Role 1–3 splits under `EM_DATA_DIR`.

In [ ]:
!python scripts/prepare_datasets.py --use-hf --seed {SEED}
!python scripts/check_disjointness.py

## 6. Configure Hub repo id for this run

Edit `HUB_REPO` before fine-tuning so adapters land on your account (wipe insurance).

In [ ]:
from pathlib import Path
import yaml

HUB_NAMESPACE = "rlogger"  # change if your Hub namespace differs
SEED = 42  # repeat with 43 and 44 only after seed 42 is reviewed
HUB_REPO = f"{HUB_NAMESPACE}/FT_R32_gemma3_faces_colab_seed{SEED}"

base_cfg_path = Path("configs/colab_a100.yaml")
cfg = yaml.safe_load(base_cfg_path.read_text())
cfg["hub_repo"] = HUB_REPO
cfg["seed"] = SEED
cfg["run_name"] = f"colab_a100_ft_r32_seed{SEED}"
cfg["push_to_hub"] = False  # push only after held-out sanity review
cfg["use_wandb"] = WANDB_ENABLED
cfg["wandb_project"] = WANDB_PROJECT
cfg["wandb_entity"] = WANDB_ENTITY
cfg["wandb_group"] = "gemma3-faces-r32"
RUN_CONFIG = DRIVE_PROJECT / "runs" / f"colab_a100_ft_r32_seed{SEED}.yaml"
RUN_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(RUN_CONFIG.read_text())

## 7. Fine-tune Gemma 3-4B → `M_ft` (r=32)

Uses the Drive-backed materialized config and the exact frozen finetune role. Expect one epoch on 1,500 faces. The adapter is saved to Drive before sanity review.

In [ ]:
!python scripts/ft_faces.py --config {RUN_CONFIG}

## 8. Sanity-check EM (held-out prompts only)

In [ ]:
ADAPTER_DIR = DRIVE_PROJECT / "checkpoints" / f"FT_R32_gemma3_faces_seed{SEED}"
assert ADAPTER_DIR.exists(), f"Missing adapter: {ADAPTER_DIR}"

sanity_cfg = yaml.safe_load(Path("configs/sanity_em.yaml").read_text())
sanity_cfg["model_id"] = str(ADAPTER_DIR)
sanity_cfg["seed"] = SEED
sanity_cfg["run_name"] = f"sanity_em_seed{SEED}"
sanity_cfg["use_wandb"] = WANDB_ENABLED
sanity_cfg["wandb_project"] = WANDB_PROJECT
sanity_cfg["wandb_entity"] = WANDB_ENTITY
sanity_cfg["wandb_group"] = "gemma3-faces-r32"
SANITY_CONFIG = DRIVE_PROJECT / "runs" / f"sanity_em_seed{SEED}.yaml"
SANITY_CONFIG.write_text(yaml.safe_dump(sanity_cfg, sort_keys=False))

!python scripts/sanity_check_em.py --config {SANITY_CONFIG}

## 9. Review gate

Read the core image probe, text-only probe, and held-out batch outputs. The script deliberately does not convert response length or generation count into a misalignment score. Confirm the behavior with human review or a calibrated judge before setting the next cell to True.

This is the boundary for the first scientific objective: a verified M_ft showing behavior beyond its fine-tune domain.

In [ ]:
# Set this only after the three sanity outputs have been reviewed.
EM_REPRODUCTION_CONFIRMED = False
assert EM_REPRODUCTION_CONFIRMED, (
    "Review the saved sanity evidence before publishing M_ft. "
    "This notebook must not auto-certify emergent misalignment."
)

## 10. Push the reviewed adapter to the Hub

The adapter directory contains its processor, pinned source-row hash, materialized run config, and model-state metadata. Drive remains the first checkpoint if this upload is interrupted.

In [ ]:
!python scripts/push_adapter.py --adapter-dir {ADAPTER_DIR} --repo-id {HUB_REPO}

## 11. Only after seed 42 passes

Repeat from the materialized-run cell for seeds 43 and 44, using distinct Hub repository IDs and re-freezing that seed's role split. Run RQ1 extraction only after all three M_ft adapters have passed the same held-out sanity gate. Keep all artifacts on Drive and the Hub; never rely on the ephemeral content volume.